<a href="https://colab.research.google.com/github/sahruulmubarokk-dot/Komputasi-Numerik/blob/main/SPL_3_Gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np

# SPL:
#  2x +  y -  z =   8
# -3x -  y + 2z = -11
# -2x +  y + 2z =  -3
A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)

b = np.array([8, -11, -3], dtype=float)

Cell 1: Import dan data SPL

numpy dipakai untuk operasi matriks. A adalah matriks koefisien 3×3 dan b adalah vektor konstanta. dtype=float penting supaya pembagian tidak terpotong jadi bilangan bulat. Untuk soal lain, cukup ganti isi A dan b di cell ini.

In [10]:
def gauss(A, b, tol=1e-12):
    n = len(b)

    # Langkah 3: gabungkan jadi matriks augmented [A|b] (3x4)
    M = np.hstack((A, b.reshape(-1, 1)))

    # Langkah 4: eliminasi maju
    for k in range(n - 1):
        # Jika pivot 0, cari baris di bawahnya yang pivotnya tidak 0
        if abs(M[k, k]) < tol:
            tukar = None
            for i in range(k + 1, n):
                if abs(M[i, k]) > tol:
                    tukar = i
                    break
            if tukar is None:
                return None  # tidak ada solusi tunggal
            M[[k, tukar]] = M[[tukar, k]]  # tukar baris

        # Nolkan elemen di bawah pivot
        for i in range(k + 1, n):
            faktor = M[i, k] / M[k, k]
            M[i, k:] = M[i, k:] - faktor * M[k, k:]

    # Cek pivot terakhir (a33')
    if abs(M[n - 1, n - 1]) < tol:
        return None  # tidak ada solusi tunggal

    # Langkah 5: substitusi mundur
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (M[i, n] - np.dot(M[i, i + 1:n], x[i + 1:])) / M[i, i]

    return x, M

Cell 2: Fungsi eliminasi gauss

Cell ini inti algoritma, dan urutannya sama dengan algoritma deskriptif:

np.hstack menggabungkan A dan b menjadi matriks augmented.
Loop k adalah eliminasi maju. Kalau pivot mendekati 0 (< tol, bukan == 0, karena bilangan desimal tidak selalu persis 0), baris ditukar dengan baris di bawahnya yang pivotnya tidak 0. Kalau tidak ada, fungsi mengembalikan None.
faktor dan baris M[i, k:] = ... adalah operasi “kurangkan baris i dengan faktor × baris pivot”.
Setelah loop, pivot terakhir dicek (perbaikan yang tadi kita bahas).
Substitusi mundur dimulai dari z, lalu y, lalu x. Rumus umumnya satu loop yang berjalan dari baris terakhir ke atas.
Fungsi mengembalikan solusi x dan matriks hasil eliminasi M.

In [11]:
hasil = gauss(A, b)

if hasil is None:
    print("SPL tidak memiliki solusi tunggal.")
else:
    x, M = hasil
    print("Matriks augmented setelah eliminasi maju:")
    print(np.round(M, 4))
    print()
    print(f"x = {x[0]:.4f}")
    print(f"y = {x[1]:.4f}")
    print(f"z = {x[2]:.4f}")

Matriks augmented setelah eliminasi maju:
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   0.  -1.   1. ]]

x = 2.0000
y = 3.0000
z = -1.0000


Cell 3: Jalankan dan tampilkan hasil

Fungsi gauss dipanggil. Kalau hasilnya None, program memberi tahu bahwa tidak ada solusi tunggal. Kalau tidak, program mencetak matriks segitiga atas (bagus untuk screenshot laporan) dan nilai x, y, z. Untuk SPL contoh ini, hasilnya x = 2, y = 3, z = −1.

In [12]:
# Cek 1: substitusi balik, A @ x harus sama dengan b
print("A @ x =", np.round(A @ x, 4))
print("b     =", b)

# Cek 2: bandingkan dengan fungsi bawaan NumPy
x_numpy = np.linalg.solve(A, b)
print("Solusi NumPy:", np.round(x_numpy, 4))
print("Sama dengan hasil Gauss?", np.allclose(x, x_numpy))

A @ x = [  8. -11.  -3.]
b     = [  8. -11.  -3.]
Solusi NumPy: [ 2.  3. -1.]
Sama dengan hasil Gauss? True


Cell 4 : Verifikasi Hasil

Cell ini membuktikan hasilmu benar. Cek pertama mengalikan A dengan x, dan hasilnya harus sama dengan b. Cek kedua membandingkan dengan np.linalg.solve. np.allclose dipakai (bukan ==) karena selisih pembulatan desimal yang sangat kecil dianggap sama.

In [13]:
# Baris 3 = baris 1 + baris 2, jadi persamaannya saling bergantung
A2 = np.array([[1, 2, 3],
               [2, 1, 1],
               [3, 3, 4]], dtype=float)
b2 = np.array([6, 5, 11], dtype=float)

hasil2 = gauss(A2, b2)
print("Tidak ada solusi tunggal" if hasil2 is None else hasil2[0])

Tidak ada solusi tunggal


Cell 5 : Uji kasus tanpa solusi tunggal

Cell ini menguji pengecekan pivot. Baris ketiga adalah jumlah baris pertama dan kedua, sehingga setelah eliminasi pivot terakhir menjadi 0 dan program berhenti dengan pesan yang benar, tanpa error pembagian dengan nol. Cell ini bagus untuk menunjukkan ke dosen bahwa programmu menangani kasus khusus.